<a href="https://colab.research.google.com/github/RewelliOliveira/review-distill-ptbr/blob/main/notebooks/02_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Fine-tuning the student (LoRA on Qwen2.5-0.5B-Instruct)

Runs on **Google Colab with a free T4 GPU** (*Runtime → Change runtime type → T4 GPU*).

The student learns to map a review (chat format) to the teacher's JSON diagnosis:
`system instruction + review text → {"aspects": ..., "main_issue": ..., "needs_contact": ...}`.
All logic lives in `src/review_distill/`; this notebook only orchestrates.

Steps: clone the repo → install the `train` extra → load `train.jsonl` / `val.jsonl` →
LoRA fine-tuning → generation-based evaluation on **validation** → raw predictions on the
**gold set** scored by `scripts/evaluate.py` → save the adapter (Drive / download / Hub).

## Inputs: `train.jsonl` and `val.jsonl`

The teacher labels are not versioned in git, so the two files must be provided. Create them
locally with `uv run python scripts/make_splits.py` (they land in `data/labeled/`), then pick
one option in the config cell:

- **`DATA_SOURCE = "upload"`** — the data cell opens a file picker; select both files.
  Simple, but must be repeated every session, and the outputs are only downloaded at the end.
- **`DATA_SOURCE = "drive"`** — copy both files once to a Google Drive folder
  (default `MyDrive/review-distill/`). The notebook mounts Drive (asks for authorization),
  reads the files from there and also saves the adapter and predictions there,
  so nothing is lost if the Colab session disconnects.

The gold set (`data/gold/`) is in git, so it comes with the clone.

## Hugging Face token

Add a secret named `HF_TOKEN` in Colab (🔑 icon in the left sidebar → *Add new secret*,
enable *Notebook access*). It is read with `google.colab.userdata`, never written in the
notebook. It is optional for training (the base model is public) and required only for
`PUSH_TO_HUB = True` (use a token with *write* access).

In [1]:
# ---- configuration ----
REPO_URL = "https://github.com/RewelliOliveira/review-distill-ptbr.git"
BRANCH = "main"  # branch with the training code; switch to "main" once merged

DATA_SOURCE = "drive"  # "upload" or "drive"
DRIVE_DIR = "/content/drive/MyDrive/review-distill"  # used when DATA_SOURCE == "drive"

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
RUN_NAME = "qwen2.5-0.5b-lora"  # results/<RUN_NAME>.json
SEED = 42

PUSH_TO_HUB = False
HUB_REPO_ID = ""  # e.g. "<your-hf-user>/review-distill-qwen2.5-0.5b-lora"

## Setup: clone the repository and install the `train` extra

In [2]:
import os
import sys

REPO_DIR = "/content/review-distill-ptbr"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

!pip install -q uv
!uv pip install --system -q -e ".[train]"
sys.path.insert(0, f"{REPO_DIR}/src")  # the editable install is only seen after a restart

!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/content/review-distill-ptbr'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 58 (delta 3), reused 51 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 400.25 KiB | 7.28 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/review-distill-ptbr
6fd91c5 (grafted, HEAD -> main, origin/main, origin/HEAD) Merge pull request #1 from RewelliOliveira/developer
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception as e:  # secret missing or notebook access not granted
    print(f"HF_TOKEN not available ({type(e).__name__}); fine unless PUSH_TO_HUB is True")
    if PUSH_TO_HUB:
        raise

HF_TOKEN not available (SecretNotFoundError); fine unless PUSH_TO_HUB is True


## Data: `train.jsonl` and `val.jsonl`

In [4]:
import shutil
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)
SPLIT_FILES = ("train.jsonl", "val.jsonl")

if DATA_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path(DRIVE_DIR) / RUN_NAME
    for name in SPLIT_FILES:
        shutil.copy(Path(DRIVE_DIR) / name, DATA_DIR / name)
elif DATA_SOURCE == "upload":
    from google.colab import files

    OUT_DIR = Path("/content/outputs") / RUN_NAME
    print("Select train.jsonl and val.jsonl")
    for name, content in files.upload().items():
        # Colab renames repeated uploads ("train (1).jsonl"); map back by prefix
        target = next((s for s in SPLIT_FILES if name.startswith(s.split(".")[0])), name)
        (DATA_DIR / target).write_bytes(content)
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
missing = [n for n in SPLIT_FILES if not (DATA_DIR / n).exists()]
assert not missing, f"missing {missing} in {DATA_DIR}"
print("outputs ->", OUT_DIR)

Mounted at /content/drive
outputs -> /content/drive/MyDrive/review-distill/qwen2.5-0.5b-lora


In [5]:
import pandas as pd
from datasets import Dataset
from transformers import set_seed

from review_distill.gold import diagnosis_from_record
from review_distill.inference import STUDENT_PROMPT_VERSION, training_example
from review_distill.sampling import GOLD_REVIEWS_PATH, assert_disjoint, load_sample
from review_distill.splits import load_split

set_seed(SEED)
train_df = load_split(DATA_DIR / "train.jsonl")
val_df = load_split(DATA_DIR / "val.jsonl")
gold_df = load_sample(GOLD_REVIEWS_PATH)
assert_disjoint(gold_df, pd.concat([train_df, val_df]))  # no gold leakage


def diagnoses(df):
    return [diagnosis_from_record(r) for r in df.to_dict("records")]


def to_dataset(df):
    rows = [training_example(t, d) for t, d in zip(df["text"], diagnoses(df), strict=True)]
    return Dataset.from_list(rows)


train_ds, val_ds = to_dataset(train_df), to_dataset(val_df)
print(f"train {len(train_ds)} | val {len(val_ds)} | gold {len(gold_df)}")
train_ds[0]

train 1800 | val 200 | gold 150


{'prompt': [{'content': 'You extract a structured diagnosis from a Brazilian e-commerce customer review (in Portuguese). Answer only with a JSON object with the keys aspects (delivery_time, product_quality, as_described, order_accuracy, packaging, customer_service: positive | negative | not_mentioned), main_issue and needs_contact.',
   'role': 'system'},
  {'content': 'não recomendo. Comprei 6 tapetes e só recebi 4. Estão faltando dois.',
   'role': 'user'}],
 'completion': [{'content': '{"aspects": {"delivery_time": "not_mentioned", "product_quality": "not_mentioned", "as_described": "not_mentioned", "order_accuracy": "negative", "packaging": "not_mentioned", "customer_service": "not_mentioned"}, "main_issue": "missing_item", "needs_contact": true}',
   'role': 'assistant'}]}

## LoRA fine-tuning

The base model is loaded in fp32 (0.5B fits easily on a T4) and trained with fp16 mixed
precision; only the LoRA adapters are trained. The loss is computed on the completion
(the JSON) only. Checkpoints are compared by validation loss after each epoch.
Hyperparameters were fixed up front; any change must be judged on validation, never on gold.

In [6]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

HPARAMS = {
    "num_train_epochs": 3,
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 10,
    "weight_decay": 0.0,
    "max_length": 512,
}
LORA = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": "all-linear",
}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to("cuda")

args = SFTConfig(
    output_dir="/content/checkpoints",
    **HPARAMS,
    per_device_eval_batch_size=32,
    fp16=True,  # T4 has no bf16 support
    bf16=False,  # recent TRL enables bf16 by default
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=1,
    seed=SEED,
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=LoraConfig(task_type="CAUSAL_LM", **LORA),
)
trainer.model.print_trainable_parameters()
train_result = trainer.train()
train_result.metrics

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

## Save the adapter

In [ ]:
ADAPTER_DIR = OUT_DIR / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to", ADAPTER_DIR)
!ls -la {ADAPTER_DIR}

## Evaluation on validation (generation)

Greedy decoding, outputs parsed with the same parser used by `scripts/evaluate.py`.
For inference the adapter is merged into the base model and cast to fp16.

In [ ]:
import json

from review_distill.inference import generate_raw, parse_diagnosis
from review_distill.metrics import compute_metrics, latency_stats

GENERATION = {"batch_size": 16, "max_new_tokens": 128, "do_sample": False}

infer_model = trainer.model.merge_and_unload().half().eval()
val_out = generate_raw(
    infer_model,
    tokenizer,
    list(val_df["text"]),
    batch_size=GENERATION["batch_size"],
    max_new_tokens=GENERATION["max_new_tokens"],
)
val_metrics = compute_metrics(diagnoses(val_df), [parse_diagnosis(raw) for raw, _ in val_out])
val_metrics |= latency_stats([s for _, s in val_out])
print(json.dumps(val_metrics, indent=2))

In [ ]:
# Invalid outputs on validation, if any (for debugging the format)
bad = [
    (t, raw)
    for t, (raw, _) in zip(val_df["text"], val_out, strict=True)
    if parse_diagnosis(raw) is None
]
print(f"{len(bad)} invalid outputs")
for text, raw in bad[:5]:
    print("-", text, "\n  ->", raw)

## Predictions on the gold set

Raw outputs are saved as `results/predictions/<RUN_NAME>_raw.jsonl` and scored by
`scripts/evaluate.py`, the same script used for the teacher and the TF-IDF baseline.
These numbers are the final test: **do not tune anything based on them.**

In [ ]:
import platform

import peft
import transformers
import trl

from review_distill.inference import prediction_record

PRED_DIR = Path(REPO_DIR) / "results" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
raw_path = PRED_DIR / f"{RUN_NAME}_raw.jsonl"
config_path = PRED_DIR / f"{RUN_NAME}_config.json"

gold_out = generate_raw(
    infer_model,
    tokenizer,
    list(gold_df["text"]),
    batch_size=GENERATION["batch_size"],
    max_new_tokens=GENERATION["max_new_tokens"],
)
with raw_path.open("w", encoding="utf-8") as f:
    for rid, (raw, seconds) in zip(gold_df["review_id"], gold_out, strict=True):
        f.write(json.dumps(prediction_record(rid, raw, seconds), ensure_ascii=False) + "\n")

run_config = {
    "base_model": BASE_MODEL,
    "student_prompt_version": STUDENT_PROMPT_VERSION,
    "seed": SEED,
    "hparams": HPARAMS,
    "lora": LORA,
    "precision": "fp32 weights + fp16 mixed precision (train); merged fp16 (inference)",
    "generation": GENERATION,
    "latency_mode": f"batched ({GENERATION['batch_size']}/batch), {torch.cuda.get_device_name()}",
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "train_metrics": train_result.metrics,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "val_metrics": val_metrics,
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "peft": peft.__version__,
    },
}
config_path.write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")

!python scripts/evaluate.py --model {RUN_NAME} --predictions {raw_path} --config {config_path}

## Export the outputs

Copies the evaluation files next to the adapter. To add them to the repository, put
`results/<RUN_NAME>.json` and `results/predictions/<RUN_NAME>*.jsonl|json` in `results/`
locally (or re-run `scripts/evaluate.py` locally with the raw predictions and config).

In [ ]:
RESULTS_DIR = Path(REPO_DIR) / "results"
for path in [RESULTS_DIR / f"{RUN_NAME}.json", *PRED_DIR.glob(f"{RUN_NAME}*")]:
    shutil.copy(path, OUT_DIR / path.name)
!ls -la {OUT_DIR}

if DATA_SOURCE == "upload":
    from google.colab import files

    archive = shutil.make_archive(f"/content/{RUN_NAME}", "zip", OUT_DIR)
    files.download(archive)

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    assert HUB_REPO_ID, "set HUB_REPO_ID"
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(HUB_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=HUB_REPO_ID)
    print(f"adapter pushed to https://huggingface.co/{HUB_REPO_ID} (private)")